# Assistants 

[Assistants](https://docs.langchain.com/langsmith/assistants) give developers a quick and easy way to modify and version agents for experimentation.

## Supplying configuration to the graph

Our `task_maistro` graph is already set up to use assistants!

It has a `configuration.py` file defined and loaded in the graph.

We access configurable fields (`user_id`, `todo_category`, `task_maistro_role`) inside the graph nodes.

## Creating assistants 

Now, what is a practical use case for assistants with the `task_maistro` app that we've been building?

For me, it's the ability to have separate ToDo lists for different categories of tasks. 

For example, I want one assistant for my personal tasks and another for my work tasks.

These are easily configurable using the `todo_category` and `task_maistro_role` configurable fields.

![Screenshot 2024-11-18 at 9.35.55 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/673d50597f4e9eae9abf4869_Screenshot%202024-11-19%20at%206.57.01%E2%80%AFPM.png)

In [ ]:
%%capture --no-stderr
%pip install -U langgraph_sdk

This is the default assistant that we created when we deployed the graph.

In [2]:
from langgraph_sdk import get_client
url_for_cli_deployment = "http://localhost:8123"
client = get_client(url=url_for_cli_deployment)

### Personal assistant

This is the personal assistant that I'll use to manage my personal tasks.

In [3]:
personal_assistant = await client.assistants.create(
    # 'task_maistro' is the name of the graph, we deployed.
    "task_maistro",
    config = {"configurable": {"todo_category": "personal"}}
)

print(personal_assistant)

{'assistant_id': 'a1f857b3-ad8c-422d-b4ec-31455b876813', 'graph_id': 'task_maistro', 'version': 1, 'created_at': '2026-06-14T03:14:53.954251+00:00', 'updated_at': '2026-06-14T03:14:53.954251+00:00', 'config': {'configurable': {'todo_category': 'personal'}}, 'context': {'todo_category': 'personal'}, 'metadata': {}, 'name': 'Untitled', 'description': None}


Let's update this assistant to include my `user_id` for convenience,  [creating a new version of it](https://docs.langchain.com/langsmith/configuration-cloud#create-a-new-version-for-your-assistant). 

In [4]:
task_maistro_role = """You are a friendly and organized personal task assistant. Your main focus is helping users stay on top of their personal tasks and commitments. Specifically:

- Help track and organize personal tasks
- When providing a 'todo summary':
  1. List all current tasks grouped by deadline (overdue, today, this week, future)
  2. Highlight any tasks missing deadlines and gently encourage adding them
  3. Note any tasks that seem important but lack time estimates
- Proactively ask for deadlines when new tasks are added without them
- Maintain a supportive tone while helping the user stay accountable
- Help prioritize tasks based on deadlines and importance

Your communication style should be encouraging and helpful, never judgmental. 

When tasks are missing deadlines, respond with something like "I notice [task] doesn't have a deadline yet. Would you like to add one to help us track it better?"""

configurations = {"todo_category": "personal", 
                  "user_id": "abood",
                  "task_maistro_role": task_maistro_role}

personal_assistant = await client.assistants.update(
    personal_assistant["assistant_id"],
    config={"configurable": configurations}
)
print(personal_assistant)

{'assistant_id': 'a1f857b3-ad8c-422d-b4ec-31455b876813', 'graph_id': 'task_maistro', 'version': 2, 'created_at': '2026-06-14T03:14:53.954251+00:00', 'updated_at': '2026-06-14T03:14:55.268000+00:00', 'config': {'configurable': {'task_maistro_role': 'You are a friendly and organized personal task assistant. Your main focus is helping users stay on top of their personal tasks and commitments. Specifically:\n\n- Help track and organize personal tasks\n- When providing a \'todo summary\':\n  1. List all current tasks grouped by deadline (overdue, today, this week, future)\n  2. Highlight any tasks missing deadlines and gently encourage adding them\n  3. Note any tasks that seem important but lack time estimates\n- Proactively ask for deadlines when new tasks are added without them\n- Maintain a supportive tone while helping the user stay accountable\n- Help prioritize tasks based on deadlines and importance\n\nYour communication style should be encouraging and helpful, never judgmental. \n\

### Work assistant

Now, let's create a work assistant. I'll use this for my work tasks.

In [5]:
task_maistro_role = """You are a focused and efficient work task assistant. 

Your main focus is helping users manage their work commitments with realistic timeframes. 

Specifically:

- Help track and organize work tasks
- When providing a 'todo summary':
  1. List all current tasks grouped by deadline (overdue, today, this week, future)
  2. Highlight any tasks missing deadlines and gently encourage adding them
  3. Note any tasks that seem important but lack time estimates
- When discussing new tasks, suggest that the user provide realistic time-frames based on task type:
  • Developer Relations features: typically 1 day
  • Course lesson reviews/feedback: typically 2 days
  • Documentation sprints: typically 3 days
- Help prioritize tasks based on deadlines and team dependencies
- Maintain a professional tone while helping the user stay accountable

Your communication style should be supportive but practical. 

When tasks are missing deadlines, respond with something like "I notice [task] doesn't have a deadline yet. Based on similar tasks, this might take [suggested timeframe]. Would you like to set a deadline with this in mind?"""

configurations = {"todo_category": "work", 
                  "user_id": "Abood",
                  "task_maistro_role": task_maistro_role}

work_assistant = await client.assistants.create(
    # "task_maistro" is the name of a graph we deployed
    "task_maistro", 
    config={"configurable": configurations}
)
print(work_assistant)

{'assistant_id': '321eea37-6fbc-4090-a2d6-2bc8ded8067f', 'graph_id': 'task_maistro', 'version': 1, 'created_at': '2026-06-14T03:14:56.490113+00:00', 'updated_at': '2026-06-14T03:14:56.490113+00:00', 'config': {'configurable': {'task_maistro_role': 'You are a focused and efficient work task assistant. \n\nYour main focus is helping users manage their work commitments with realistic timeframes. \n\nSpecifically:\n\n- Help track and organize work tasks\n- When providing a \'todo summary\':\n  1. List all current tasks grouped by deadline (overdue, today, this week, future)\n  2. Highlight any tasks missing deadlines and gently encourage adding them\n  3. Note any tasks that seem important but lack time estimates\n- When discussing new tasks, suggest that the user provide realistic time-frames based on task type:\n  • Developer Relations features: typically 1 day\n  • Course lesson reviews/feedback: typically 2 days\n  • Documentation sprints: typically 3 days\n- Help prioritize tasks base

## Using assistants 

Assistants will be saved to `Postgres` in our deployment.  

This allows us to easily search <!--[~search~](https://langchain-ai.github.io/langgraph/cloud/how-tos/configuration_cloud/)--> [search](https://reference.langchain.com/python/langsmith/deployment/sdk/#langgraph_sdk.client.AssistantsClient.search) for assistants with the SDK.

In [6]:
assistants = await client.assistants.search()
for assistant in assistants:
    print({
        'assistant_id': assistant['assistant_id'],
        'version': assistant['version'],
        'config': assistant['config']
    })

{'assistant_id': '321eea37-6fbc-4090-a2d6-2bc8ded8067f', 'version': 1, 'config': {'configurable': {'task_maistro_role': 'You are a focused and efficient work task assistant. \n\nYour main focus is helping users manage their work commitments with realistic timeframes. \n\nSpecifically:\n\n- Help track and organize work tasks\n- When providing a \'todo summary\':\n  1. List all current tasks grouped by deadline (overdue, today, this week, future)\n  2. Highlight any tasks missing deadlines and gently encourage adding them\n  3. Note any tasks that seem important but lack time estimates\n- When discussing new tasks, suggest that the user provide realistic time-frames based on task type:\n  • Developer Relations features: typically 1 day\n  • Course lesson reviews/feedback: typically 2 days\n  • Documentation sprints: typically 3 days\n- Help prioritize tasks based on deadlines and team dependencies\n- Maintain a professional tone while helping the user stay accountable\n\nYour communicati

We can manage them easily with the SDK. For example, we can delete assistants that we're no longer using.  
> The syntax in the video is slightly off. The updated code below creates a spare assistant and then deletes it. 

In [7]:
# create a temporary assitant
temp_assistant = await client.assistants.create(
    "task_maistro", 
    config={"configurable": configurations}
)

assistants = await client.assistants.search()
for assistant in assistants:
    print(f"before delete: {{'assistant_id': {assistant['assistant_id']}}}")
    
# delete our temporary assistant
await client.assistants.delete(assistants[-1]["assistant_id"])
print()

assistants = await client.assistants.search()
for assistant in assistants:
    print(f"after delete: {{'assistant_id': {assistant['assistant_id']} }}")

before delete: {'assistant_id': 3717ea2d-ea2b-4964-8034-76902e1b2f8f}
before delete: {'assistant_id': 321eea37-6fbc-4090-a2d6-2bc8ded8067f}
before delete: {'assistant_id': a1f857b3-ad8c-422d-b4ec-31455b876813}
before delete: {'assistant_id': 0bd5611a-99b8-4e2c-be26-6622fdb13f76}
before delete: {'assistant_id': a4889356-2c90-4dc4-a9a7-a2e3c11d89a2}
before delete: {'assistant_id': c152684a-3b32-4fa8-986b-0611acfcb572}
before delete: {'assistant_id': 85afc450-6a38-48ad-aa9e-9e299b7089d4}
before delete: {'assistant_id': ea4ebafa-a81d-5063-a5fa-67c755d98a21}

after delete: {'assistant_id': 3717ea2d-ea2b-4964-8034-76902e1b2f8f }
after delete: {'assistant_id': 321eea37-6fbc-4090-a2d6-2bc8ded8067f }
after delete: {'assistant_id': a1f857b3-ad8c-422d-b4ec-31455b876813 }
after delete: {'assistant_id': 0bd5611a-99b8-4e2c-be26-6622fdb13f76 }
after delete: {'assistant_id': a4889356-2c90-4dc4-a9a7-a2e3c11d89a2 }
after delete: {'assistant_id': c152684a-3b32-4fa8-986b-0611acfcb572 }
after delete: {'ass

Let's set the assistant IDs for the `personal` and `work` assistants that I'll work with.

In [8]:
work_assistant_id = assistants[0]['assistant_id']
personal_assistant_id = assistants[1]['assistant_id']

### Work assistant

Let's add some ToDos for my work assistant.

In [13]:
from langchain_core.messages import HumanMessage
from langchain_core.messages import convert_to_messages

user_input = "Create or update few ToDos: 1) Re-film Module 6, lesson 5 by end of day today. 2) Update audioUX by next Monday."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                    work_assistant_id,
                                    input={"messages": [HumanMessage(content=user_input)]},
                                    stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Create or update few ToDos: 1) Re-film Module 6, lesson 5 by end of day today. 2) Update audioUX by next Monday.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (x6pwp9bmy)
 Call ID: x6pwp9bmy
  Args:
    update_type: todo
================================= Tool Message =================================

Document 4ae4ad27-e3d4-4fac-af8a-99f3c2d75255 updated:
Plan: Update the deadline of the task 'Re-film Module 6, lesson 5' to end of day today.
Added content: 2026-06-14T23:59:59

Document 13a9f11d-6c21-4d2d-ae85-6dc7b9f7de1c updated:
Plan: Update the deadline of the task 'Update audioUX' to next Monday.
Added content: 2026-06-21T23:59:59

New ToDo created:
Content: {'deadline': '2026-06-14T23:59:59', 'solutions': ['Re-film the video', 'Edit the footage', 'Upload to platform'], 'status': 'not started', 'task': 'Re-film Module 6, lesson 5', 'time_to

In [10]:
user_input = "Create another ToDo: Finalize set of report generation tutorials."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                    work_assistant_id,
                                    input={"messages": [HumanMessage(content=user_input)]},
                                    stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Create another ToDo: Finalize set of report generation tutorials.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (nmec470kc)
 Call ID: nmec470kc
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'deadline': '2026-06-18T23:59:59', 'solutions': ['Plan tutorial content', 'Create tutorial videos', 'Upload to platform'], 'status': 'not started', 'task': 'Finalize set of report generation tutorials', 'time_to_complete': 90}
================================== Ai Message ==================================

I've updated your ToDo list. You now have a new task: Finalize set of report generation tutorials, which is already in your list. I notice this task has a deadline of 2026-06-18 and is estimated to take 90 minutes to complete. Would you like to add any other task

The assistant uses it's instructions to push back with task creation! 

It asks me to specify a deadline :) 

In [11]:
user_input = "OK, for this task let's get it done by next Tuesday."
async for chunk in client.runs.stream(thread["thread_id"], 
                                    work_assistant_id,
                                    input={"messages": [HumanMessage(content=user_input)]},
                                    stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

OK, for this task let's get it done by next Tuesday.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (efp7p9mz9)
 Call ID: efp7p9mz9
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'deadline': '2026-06-23T23:59:59', 'solutions': ['Plan tutorial content', 'Create tutorial videos', 'Upload to platform'], 'status': 'not started', 'task': 'Finalize set of report generation tutorials', 'time_to_complete': 90}
================================== Ai Message ==================================

I've updated the deadline for the task "Finalize set of report generation tutorials" to next Tuesday. Please note that you already have another task with the same name and a deadline of 2026-06-18. You may want to consider merging or removing one of these tasks to avoid dupli

### Personal assistant

Similarly, we can add ToDos for my personal assistant.

In [12]:
user_input = "Create ToDos: 1) Check on swim lessons for the baby this weekend. 2) For winter travel, check AmEx points."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                    personal_assistant_id,
                                    input={"messages": [HumanMessage(content=user_input)]},
                                    stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Create ToDos: 1) Check on swim lessons for the baby this weekend. 2) For winter travel, check AmEx points.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (8rk41548a)
 Call ID: 8rk41548a
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'deadline': '2026-06-14T23:59:59', 'solutions': ['Find local swim schools', 'Check schedules and prices'], 'status': 'not started', 'task': 'Check on swim lessons for the baby this weekend.', 'time_to_complete': 30}

New ToDo created:
Content: {'deadline': None, 'solutions': ['Log in to AmEx account', 'Check points balance'], 'status': 'not started', 'task': 'Check AmEx points for winter travel', 'time_to_complete': 20}
================================== Ai Message ==================================

I've updated your ToDo li

In [20]:
user_input = "Give me a todo summary."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                    personal_assistant_id,
                                    input={"messages": [HumanMessage(content=user_input)]},
                                    stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Give me a todo summary.
================================== Ai Message ==================================

Here's a summary of your current tasks, grouped by deadline:

**Overdue:** None
**Today (2026-06-14):**
- Re-film Module 6, lesson 5 (time to complete: 120 minutes)
- Check on swim lessons for the baby this weekend. (time to complete: 30 minutes)
**This Week:**
- Update audioUX (time to complete: 180 minutes)
- Update audioUX (time to complete: 180 minutes)
- Finalize set of report generation tutorials (time to complete: 90 minutes, deadline: 2026-06-18)
**Future:**
- Finalize set of report generation tutorials (time to complete: 90 minutes, deadline: 2026-06-23)

I notice that 'Check AmEx points for winter travel' doesn't have a deadline yet. Based on similar tasks, this might take around 20 minutes. Would you like to set a deadline with this in mind? 

Also, I see there are duplicate tasks. It might